[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/ab-test-practice/blob/main/study_notes/week03_study_early_results.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 3: Dealing with Early Results in A/B Tests

**Course:** A/B Testing in Practice ()  
**Company Context:** FamilyNest - Airbnb for families with young kids  
**Role:** Product Data Scientist  

---

## Scenario

FamilyNest is running an experiment on a new "Kid-Friendly Amenities" badge for listings.  
The hypothesis: highlighting amenities like cribs, high chairs, baby gates, and fenced yards will increase booking rates among families with children under 5.

The test was designed to run for 3 weeks. It's Day 5. The PM pings you:

> *"Hey! The badge variant is showing +8% bookings with p=0.03! Can we ship it early?"*

This notebook covers why the answer is almost always **"Not yet"** and how to handle the conversation.

---
## 1. The Peeking Problem

### What is it?

**Peeking** = checking your A/B test results before the pre-planned end date and making decisions based on what you see.

### Why is it dangerous?

When you run a test at alpha = 0.05, you have a 5% chance of a false positive **at the single planned analysis point**. But if you check multiple times during the test, each check is an opportunity for random noise to look significant.

**The math:**
- 1 peek: actual alpha ~ 0.05
- 2 peeks: actual alpha ~ 0.08
- 5 peeks: actual alpha ~ 0.14
- 10 peeks: actual alpha ~ 0.19
- Continuous monitoring: actual alpha ~ 0.30+

### The Coin Flip Analogy

Imagine flipping a fair coin and tracking cumulative heads vs tails. If you're allowed to **stop whenever you're ahead**, you'll almost always find a moment where heads is "significantly" beating tails -- even though the coin is perfectly fair.

That's exactly what peeking does to your experiment: you're giving yourself permission to stop at a random high point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Simulate peeking: run 1000 A/A tests (no real effect) and peek at multiple time points
n_simulations = 1000
n_total = 10000  # total users per group at end
peek_points = [1000, 2000, 3000, 5000, 7000, 10000]  # sample sizes at each peek
alpha = 0.05

false_positives_no_peek = 0
false_positives_with_peek = 0

for _ in range(n_simulations):
    # Generate full data from SAME distribution (no real effect)
    # FamilyNest context: booking rate ~ 4%
    control = np.random.binomial(1, 0.04, n_total)
    treatment = np.random.binomial(1, 0.04, n_total)  # Same rate! No effect!
    
    # No peeking: only check at end
    _, p_final = stats.ttest_ind(control, treatment)
    if p_final < alpha:
        false_positives_no_peek += 1
    
    # With peeking: check at each peek point, stop if significant
    found_sig = False
    for n in peek_points:
        _, p = stats.ttest_ind(control[:n], treatment[:n])
        if p < alpha:
            found_sig = True
            break
    if found_sig:
        false_positives_with_peek += 1

print("=== The Peeking Problem (A/A Test Simulation) ===")
print(f"\nFalse positive rate WITHOUT peeking: {false_positives_no_peek/n_simulations:.1%}")
print(f"False positive rate WITH peeking ({len(peek_points)} peeks): {false_positives_with_peek/n_simulations:.1%}")
print(f"\nExpected alpha: {alpha:.1%}")
print(f"Actual alpha with peeking: {false_positives_with_peek/n_simulations:.1%}")
print(f"\n--> Peeking inflated our false positive rate by {false_positives_with_peek/n_simulations - alpha:.1%} percentage points!")

In [ ]:
# Visualize: p-value trajectory for a single A/A test
# Shows how p-values fluctuate and can dip below 0.05 by chance

np.random.seed(7)
n_total = 10000

control = np.random.binomial(1, 0.04, n_total)
treatment = np.random.binomial(1, 0.04, n_total)  # NO real effect

sample_sizes = range(100, n_total + 1, 100)
p_values = []

for n in sample_sizes:
    _, p = stats.ttest_ind(control[:n], treatment[:n])
    p_values.append(p)

plt.figure(figsize=(10, 5))
plt.plot(list(sample_sizes), p_values, linewidth=0.8, color='steelblue')
plt.axhline(y=0.05, color='red', linestyle='--', label='alpha = 0.05')
plt.fill_between(list(sample_sizes), 0, 0.05, alpha=0.1, color='red')
plt.xlabel('Sample Size per Group')
plt.ylabel('p-value')
plt.title('FamilyNest A/A Test: p-value over time\n(No real effect exists - both groups have 4% booking rate)')
plt.legend()
plt.ylim(0, 1)
plt.annotate('Danger zone: peeking here\nwould falsely declare a winner!', 
             xy=(0, 0.025), fontsize=9, color='red',
             xytext=(2000, 0.15), arrowprops=dict(arrowstyle='->', color='red'))
plt.tight_layout()
plt.show()

### Key Takeaway

Even when there is **NO real difference** between treatment and control, p-values randomly fluctuate. If you peek often enough, you WILL find a moment where p < 0.05. Stopping at that moment means you've shipped a feature with no real effect, based on noise.

**At FamilyNest:** If we peek at the Kid-Friendly Badge experiment 5 times over 3 weeks, our actual false positive rate is ~14%, not 5%. That means a 1-in-7 chance we ship a badge that does nothing for bookings.

---
## 2. When Early Stopping IS Acceptable

Not all early stopping is bad. Here are legitimate cases:

### 2.1 Pre-Specified Interim Analyses with Alpha Spending

You can **plan** to look at results early, but you must:
- Decide the number of looks **before** the test starts
- Use an **alpha-spending function** (e.g., O'Brien-Fleming, Pocock) to allocate your alpha budget across looks
- Each interim look uses a stricter significance threshold

| Method | Look 1 (50%) | Look 2 (100%) |
|--------|-------------|---------------|
| No correction | 0.050 | 0.050 |
| O'Brien-Fleming | 0.005 | 0.048 |
| Pocock | 0.029 | 0.029 |

### 2.2 Safety Monitoring (Guardrail Metrics)

If a guardrail metric is **clearly tanking**, you should stop:
- Revenue per user dropping significantly
- Crash rate spiking
- Customer support tickets surging

**FamilyNest example:** If the badge experiment causes a 20% drop in host response rates (because hosts feel pressured), that's a safety stop.

### 2.3 Obvious Harm to Users

- Medical: adverse events in clinical trials
- Tech: data loss, security vulnerabilities, severe UX degradation

### 2.4 Sequential Testing Frameworks

Modern approaches that let you monitor continuously without inflating alpha:
- **Always Valid p-values** (mSPRT)
- **Confidence sequences** (anytime-valid confidence intervals)
- **Bayesian sequential testing**

These are valid but require **upfront commitment** to the framework. You can't retrofit them after peeking with a standard test.

In [ ]:
# Demonstrate alpha-spending: O'Brien-Fleming boundaries
# Shows how the significance threshold changes at each interim look

def obrien_fleming_boundary(n_looks, alpha=0.05):
    """Approximate O'Brien-Fleming spending function boundaries.
    
    At each look k of K total looks, the z-boundary is approximately:
    z_k = z_final / sqrt(k/K)
    """
    from scipy.stats import norm
    z_final = norm.ppf(1 - alpha/2)  # ~1.96 for alpha=0.05
    
    looks = np.arange(1, n_looks + 1)
    info_fractions = looks / n_looks
    z_boundaries = z_final / np.sqrt(info_fractions)
    p_boundaries = 2 * (1 - norm.cdf(z_boundaries))
    
    return pd.DataFrame({
        'Look': looks,
        'Info Fraction': info_fractions,
        'Z Boundary': z_boundaries,
        'P Boundary': p_boundaries
    })

# FamilyNest: Suppose we plan 3 interim analyses for the badge test
boundaries = obrien_fleming_boundary(n_looks=3, alpha=0.05)
print("=== O'Brien-Fleming Boundaries for FamilyNest Badge Test ===")
print("(3 planned looks: Week 1, Week 2, Week 3)\n")
print(boundaries.to_string(index=False))
print("\nInterpretation:")
print(f"  - At Week 1 (33% of data): need p < {boundaries.iloc[0]['P Boundary']:.4f} to stop early")
print(f"  - At Week 2 (67% of data): need p < {boundaries.iloc[1]['P Boundary']:.4f} to stop early")
print(f"  - At Week 3 (100% of data): need p < {boundaries.iloc[2]['P Boundary']:.4f} (nearly standard)")
print("\nThe PM's p=0.03 at Day 5 would NOT meet the Week 1 boundary!")

---
## 3. Stakeholder Management

### The Common Pressure

> PM: "The early results look great at +8% bookings! Why can't we just ship?"  
> Engineering: "We have another feature waiting for this experiment slot."  
> Leadership: "Our OKRs deadline is next week. Can we call it?"

### How to Push Back Professionally

**Framework: Acknowledge, Explain, Recommend**

1. **Acknowledge** their excitement and urgency
2. **Explain** the specific risks in plain language
3. **Recommend** a concrete next step with a timeline

### Key Arguments (Your Arsenal)

| Argument | Plain Language Version |
|----------|----------------------|
| Novelty effects | "Users click on new things just because they're new. The +8% might just be curiosity, not genuine preference." |
| Underpowered test / Winner's curse | "When we stop a test early, the effect size we observe is inflated. The real effect might be +2% or even 0%." |
| Regression to the mean | "Extreme results early on tend to move toward average as we collect more data." |
| Day-of-week effects | "Families book differently on weekends vs weekdays. We need full weekly cycles to get a fair picture." |
| Peeking inflation | "Checking multiple times inflates our false positive rate. Our p=0.03 might actually be p=0.10+ after correction." |

### The Money Line

> **"Shipping early risks shipping something that actually hurts us. I'd rather wait 2 more weeks and be confident than ship now and learn in 3 months that bookings actually declined."**

### Sample Response to PM

> *"Great to see the early signal! I'm cautiously optimistic. However, I'd recommend we let it run the full 3 weeks for three reasons:*
> 1. *The badge is new and shiny -- we often see novelty effects flatten out after the first week.*
> 2. *At Day 5 we only have ~30% of our planned sample. The effect size is likely inflated.*
> 3. *We haven't captured a full weekend cycle yet, and FamilyNest families heavily skew toward weekend bookings.*
>
> *If it's still strong at 3 weeks, we ship with confidence. If it fades, we saved ourselves from a bad launch."*

---
## 4. Novelty Effects Deep Dive

### Definition

**Novelty effect:** Users interact more with a new feature simply because it's new and unfamiliar, not because it's genuinely better. The increased engagement fades as users get accustomed to the change.

### FamilyNest Example

The Kid-Friendly Badge is visually distinctive -- a colorful icon that stands out on search results. Early on:
- Parents notice it because it's new and eye-catching
- They click on badged listings out of curiosity
- This drives up apparent engagement and booking rates

After a week:
- The badge becomes part of the background
- Only parents who genuinely value the amenities continue to respond to it
- The true effect emerges

### How to Detect Novelty Effects

1. **Plot relative lift over time** (by cohort hour or day since exposure)
2. **If lift decreases** as users spend more time in the experiment --> novelty effect
3. **If lift stabilizes** at a positive level --> genuine effect
4. **If lift decreases to zero** --> pure novelty, no real value

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def analyze_novelty_effect(df, metric):
    """Analyze whether a treatment effect shows novelty decay over time.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: 'hrs_to_cohort' (hours since user entered experiment),
        'variant' ('treatment' or 'control'), and the metric column.
    metric : str
        Name of the metric column to analyze.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with hours_in_experiment and relative_increase columns.
    """
    max_hours = int(df['hrs_to_cohort'].max())
    df['cohort'] = df['hrs_to_cohort'].astype('int') + 1
    results = []
    for cohort in range(1, max_hours + 1):
        cohort_df = df[df['cohort'] <= cohort]
        treatment_mean = cohort_df[cohort_df['variant'] == 'treatment'][metric].mean()
        control_mean = cohort_df[cohort_df['variant'] == 'control'][metric].mean()
        pct_diff = ((treatment_mean - control_mean) / control_mean) * 100
        results.append((cohort, pct_diff))
    
    results_df = pd.DataFrame(results, columns=['hrs_in_experiment', 'relative_increase'])
    plt.figure(figsize=(8, 4))
    sns.lineplot(x='hrs_in_experiment', y='relative_increase', data=results_df, marker='o')
    plt.ylabel("Relative Increase (%)")
    plt.xlabel("Hours in Experiment")
    plt.title(f"Novelty Effect Analysis: {metric}")
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    plt.show()
    
    return results_df

In [ ]:
def analyze_novelty_effect_days(df, metric):
    """Day-based novelty analysis for longer-running experiments.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: 'days_in_experiment' (days since user entered experiment),
        'variant' ('treatment' or 'control'), and the metric column.
    metric : str
        Name of the metric column to analyze.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with day and relative_increase columns.
    """
    max_days = int(df['days_in_experiment'].max())
    results = []
    
    for day in range(1, max_days + 1):
        day_df = df[df['days_in_experiment'] <= day]
        treatment_mean = day_df[day_df['variant'] == 'treatment'][metric].mean()
        control_mean = day_df[day_df['variant'] == 'control'][metric].mean()
        if control_mean > 0:
            pct_diff = ((treatment_mean - control_mean) / control_mean) * 100
        else:
            pct_diff = 0.0
        results.append((day, pct_diff))
    
    results_df = pd.DataFrame(results, columns=['day', 'relative_increase'])
    
    plt.figure(figsize=(10, 5))
    sns.lineplot(x='day', y='relative_increase', data=results_df, marker='o', color='coral')
    plt.ylabel("Relative Increase (%)")
    plt.xlabel("Days in Experiment")
    plt.title(f"Novelty Effect Analysis (Daily): {metric}")
    plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    # Annotate interpretation zones
    if len(results_df) > 3:
        early_lift = results_df['relative_increase'].iloc[:3].mean()
        late_lift = results_df['relative_increase'].iloc[-3:].mean()
        if early_lift > 0 and late_lift < early_lift * 0.5:
            plt.annotate('Likely novelty effect!\nLift decaying over time', 
                        xy=(max_days//2, (early_lift + late_lift)/2),
                        fontsize=10, color='red', fontweight='bold')
        elif early_lift > 0 and late_lift >= early_lift * 0.7:
            plt.annotate('Lift appears stable\nLikely genuine effect', 
                        xy=(max_days//2, late_lift),
                        fontsize=10, color='green', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return results_df

In [ ]:
# Simulate FamilyNest data: Badge experiment WITH novelty effect
np.random.seed(42)

n_users_per_group = 500
max_days = 21  # 3-week experiment

# Simulate daily booking behavior
data = []

for variant in ['control', 'treatment']:
    for user_id in range(n_users_per_group):
        for day in range(1, max_days + 1):
            base_rate = 0.04  # 4% daily booking probability
            
            if variant == 'treatment':
                # Novelty effect: big boost early, decays to small genuine effect
                novelty_boost = 0.06 * np.exp(-0.2 * day)  # Decays exponentially
                genuine_effect = 0.008  # Small but real long-term lift
                rate = base_rate + novelty_boost + genuine_effect
            else:
                rate = base_rate
            
            booked = np.random.binomial(1, min(rate, 1.0))
            data.append({
                'user_id': f"{variant}_{user_id}",
                'variant': variant,
                'days_in_experiment': day,
                'hrs_to_cohort': day * 24,
                'booked': booked
            })

df_badge = pd.DataFrame(data)

print("Simulated FamilyNest Badge Experiment")
print(f"Users per group: {n_users_per_group}")
print(f"Duration: {max_days} days")
print(f"\nOverall booking rates:")
print(df_badge.groupby('variant')['booked'].mean())

In [ ]:
# Run novelty analysis on our simulated data
print("=== Novelty Effect Analysis: Kid-Friendly Badge ===")
print("Expectation: Large early lift that decays to a small genuine effect\n")

results = analyze_novelty_effect_days(df_badge, 'booked')

print("\nRelative lift by day (first 5 and last 5 days):")
print(results.head().to_string(index=False))
print("...")
print(results.tail().to_string(index=False))
print(f"\nDay 1 lift: {results.iloc[0]['relative_increase']:.1f}%")
print(f"Day 21 lift: {results.iloc[-1]['relative_increase']:.1f}%")
print("\n--> The early +8% claim was largely novelty!")

---
## 5. Underpowered Tests and Winner's Curse

### The Problem

If you stop a test early (before reaching your planned sample size), the test is **underpowered**. An underpowered test that happens to show significance has a specific pathology:

**Winner's Curse:** Among underpowered tests that reach significance, the observed effect size is systematically inflated.

### Why?

- Your test needs a large effect to cross the significance threshold with limited data
- The only way to get significance with small samples is if noise happens to push the effect UP
- So significant results from underpowered tests are biased upward

### FamilyNest Example

- **Planned:** 3-week test, MDE = 3% relative lift in bookings
- **Stopped at Day 5:** Observed +8% lift, p = 0.03
- **Reality:** The true effect might be +2% (genuine but below our MDE)
- **What happened:** Random noise inflated a small effect to look large in a small sample

### The Rule

> **You can only trust effect sizes at or above your MDE.**
> 
> If your MDE was 3%, and you observe +8% at low power, the +8% is unreliable. Wait for full power, then trust what you see.

In [ ]:
# Demonstrate Winner's Curse
# Run many underpowered experiments with a small true effect
# Among those that reach significance, show the observed effect is inflated

np.random.seed(123)

true_effect = 0.002  # True lift: 0.2 percentage points (small but real)
base_rate = 0.04
n_per_group_underpowered = 1000  # Way too small for this effect size
n_per_group_powered = 50000  # Properly powered
n_sims = 2000

observed_effects_underpowered_sig = []
observed_effects_powered_sig = []

for _ in range(n_sims):
    # Underpowered test
    control = np.random.binomial(1, base_rate, n_per_group_underpowered)
    treatment = np.random.binomial(1, base_rate + true_effect, n_per_group_underpowered)
    _, p = stats.ttest_ind(treatment, control)
    if p < 0.05:
        obs_effect = treatment.mean() - control.mean()
        observed_effects_underpowered_sig.append(obs_effect)

for _ in range(n_sims):
    # Properly powered test
    control = np.random.binomial(1, base_rate, n_per_group_powered)
    treatment = np.random.binomial(1, base_rate + true_effect, n_per_group_powered)
    _, p = stats.ttest_ind(treatment, control)
    if p < 0.05:
        obs_effect = treatment.mean() - control.mean()
        observed_effects_powered_sig.append(obs_effect)

print("=== Winner's Curse Demonstration ===")
print(f"\nTrue effect: {true_effect:.4f} ({true_effect/base_rate*100:.1f}% relative lift)")
print(f"\nUnderpowered tests (n={n_per_group_underpowered}/group):")
print(f"  Reached significance: {len(observed_effects_underpowered_sig)}/{n_sims} ({len(observed_effects_underpowered_sig)/n_sims*100:.1f}%)")
if observed_effects_underpowered_sig:
    mean_obs = np.mean(observed_effects_underpowered_sig)
    print(f"  Mean observed effect (among significant): {mean_obs:.4f} ({mean_obs/base_rate*100:.1f}% relative)")
    print(f"  Inflation factor: {mean_obs/true_effect:.1f}x the true effect!")

print(f"\nProperly powered tests (n={n_per_group_powered}/group):")
print(f"  Reached significance: {len(observed_effects_powered_sig)}/{n_sims} ({len(observed_effects_powered_sig)/n_sims*100:.1f}%)")
if observed_effects_powered_sig:
    mean_obs = np.mean(observed_effects_powered_sig)
    print(f"  Mean observed effect (among significant): {mean_obs:.4f} ({mean_obs/base_rate*100:.1f}% relative)")
    print(f"  Inflation factor: {mean_obs/true_effect:.1f}x the true effect")

print("\n--> Underpowered significant results massively overestimate the true effect!")

---
## 6. Handling Inconclusive Results

### The Misinterpretation

p > 0.05 does **NOT** mean:
- "The feature has no effect"
- "Treatment equals control"
- "The experiment failed"

p > 0.05 **DOES** mean:
- "We don't have enough evidence to conclude the feature helps"
- "The observed difference is consistent with random chance"

### Options When Your Test is Inconclusive

| Option | When to Use | FamilyNest Example |
|--------|------------|----------------------|
| **Run longer** | If you believe the effect is real but small, and you have traffic | Extend badge test 2 more weeks to detect a 2% lift |
| **Accept null, move on** | If the CI includes effects too small to matter | Badge CI is [-1%, +3%] -- even if real, max 3% isn't worth the eng cost |
| **Iterate and re-test** | If you believe the concept is right but execution was off | Redesign the badge to be more informative, test again |

### How to Communicate

**Bad:** "The experiment failed. The badge doesn't work."

**Good:** "The test did not reach statistical significance after 3 weeks (p=0.12, 95% CI: [-0.5%, +4.2%]). We cannot confidently conclude the badge increases bookings. Options: (1) extend for 2 more weeks to detect smaller effects, (2) iterate on the badge design, (3) prioritize other initiatives."

### Key Principle: Confidence Intervals Tell the Story

Always report the confidence interval, not just the p-value:
- CI of [-1%, +3%]: Small effect possible, maybe not worth pursuing
- CI of [-2%, +10%]: Wide interval, too underpowered to conclude anything
- CI of [+0.1%, +6%]: Barely missed significance, likely real but small

In [ ]:
# Demonstrate how to report inconclusive results

def report_experiment_results(control_data, treatment_data, metric_name, mde_pct=None):
    """Generate a clear experiment results summary.
    
    Parameters
    ----------
    control_data : array-like
        Metric values for control group.
    treatment_data : array-like
        Metric values for treatment group.
    metric_name : str
        Name of the metric being tested.
    mde_pct : float, optional
        Minimum detectable effect as a percentage.
    """
    control_mean = np.mean(control_data)
    treatment_mean = np.mean(treatment_data)
    
    # Absolute and relative differences
    abs_diff = treatment_mean - control_mean
    rel_diff_pct = (abs_diff / control_mean) * 100 if control_mean != 0 else 0
    
    # Statistical test
    t_stat, p_value = stats.ttest_ind(treatment_data, control_data)
    
    # Confidence interval for the difference
    se = np.sqrt(np.var(control_data)/len(control_data) + np.var(treatment_data)/len(treatment_data))
    ci_lower = abs_diff - 1.96 * se
    ci_upper = abs_diff + 1.96 * se
    ci_lower_pct = (ci_lower / control_mean) * 100 if control_mean != 0 else 0
    ci_upper_pct = (ci_upper / control_mean) * 100 if control_mean != 0 else 0
    
    print(f"{'='*60}")
    print(f"EXPERIMENT RESULTS: {metric_name}")
    print(f"{'='*60}")
    print(f"\nControl mean:    {control_mean:.4f} (n={len(control_data)})")
    print(f"Treatment mean:  {treatment_mean:.4f} (n={len(treatment_data)})")
    print(f"\nAbsolute diff:   {abs_diff:+.4f}")
    print(f"Relative diff:   {rel_diff_pct:+.1f}%")
    print(f"95% CI:          [{ci_lower_pct:+.1f}%, {ci_upper_pct:+.1f}%]")
    print(f"p-value:         {p_value:.4f}")
    
    print(f"\n{'─'*60}")
    print("INTERPRETATION:")
    if p_value < 0.05:
        print(f"  SIGNIFICANT at alpha=0.05")
        if rel_diff_pct > 0:
            print(f"  Treatment OUTPERFORMS control by {rel_diff_pct:.1f}%")
        else:
            print(f"  Treatment UNDERPERFORMS control by {abs(rel_diff_pct):.1f}%")
    else:
        print(f"  NOT SIGNIFICANT at alpha=0.05")
        print(f"  Cannot conclude treatment differs from control.")
        if mde_pct:
            if ci_upper_pct < mde_pct:
                print(f"  CI upper bound ({ci_upper_pct:+.1f}%) < MDE ({mde_pct}%) --> effect likely too small to matter")
            else:
                print(f"  CI is wide -- consider extending the test for more power")
    print(f"{'='*60}")

# Example: FamilyNest badge test -- inconclusive result
np.random.seed(99)
control = np.random.binomial(1, 0.040, 5000)
treatment = np.random.binomial(1, 0.042, 5000)  # Small real effect of +0.2pp

report_experiment_results(control, treatment, 
                          metric_name="Booking Rate (Kid-Friendly Badge)",
                          mde_pct=3.0)

---
## 7. Decision Framework with Early Results

Use this framework when you're mid-experiment and facing pressure to act:

| Scenario | Action | Rationale |
|----------|--------|-----------|
| Early results stat-sig positive, test still running | **Keep running** unless safety concern | Novelty effects, winner's curse, peeking inflation |
| Early results negative on guardrail metric | **Consider stopping** for safety | User harm outweighs learning |
| Stakeholder wants to ship early | **Push back** with novelty/power arguments | Protect long-term credibility of experimentation |
| Test ran full duration, barely significant (p~0.04) | **Launch cautiously**, monitor post-launch | Effect is real but may be small; watch for regression |
| Test ran full duration, not significant | **Don't launch**, iterate | Insufficient evidence of benefit |
| Test ran full duration, large significant effect | **Ship with confidence** | This is what we designed the test for |
| Very early + extreme results (positive or negative) | **Check for bugs first** | Instrumentation errors are more common than 20%+ effects |

In [ ]:
def early_results_decision(p_value, relative_lift_pct, days_elapsed, planned_days, 
                           is_guardrail_negative=False, sample_fraction=None):
    """Decision framework for handling early A/B test results.
    
    Parameters
    ----------
    p_value : float
        Current p-value.
    relative_lift_pct : float
        Observed relative lift in percentage.
    days_elapsed : int
        Days since experiment started.
    planned_days : int
        Total planned experiment duration in days.
    is_guardrail_negative : bool
        Whether guardrail metrics are showing negative signal.
    sample_fraction : float, optional
        Fraction of planned sample collected. If None, estimated from days.
    """
    if sample_fraction is None:
        sample_fraction = days_elapsed / planned_days
    
    print(f"{'='*60}")
    print(f"EARLY RESULTS DECISION FRAMEWORK")
    print(f"{'='*60}")
    print(f"\nCurrent State:")
    print(f"  Days elapsed:     {days_elapsed}/{planned_days} ({sample_fraction:.0%} of planned duration)")
    print(f"  Observed lift:    {relative_lift_pct:+.1f}%")
    print(f"  p-value:          {p_value:.4f}")
    print(f"  Guardrail issue:  {'YES' if is_guardrail_negative else 'No'}")
    
    print(f"\n{'─'*60}")
    print("RECOMMENDATION:")
    
    # Decision logic
    if is_guardrail_negative and p_value < 0.01:
        print("  >>> STOP THE EXPERIMENT <<<")
        print("  Guardrail metric is significantly negative.")
        print("  Action: Roll back treatment, investigate root cause.")
    elif sample_fraction >= 0.95:
        # Test is essentially complete
        if p_value < 0.05 and relative_lift_pct > 0:
            print("  >>> SHIP <<<")
            print("  Test completed with significant positive result.")
        elif p_value < 0.05 and relative_lift_pct < 0:
            print("  >>> DO NOT SHIP <<<")
            print("  Test completed with significant NEGATIVE result.")
        else:
            print("  >>> INCONCLUSIVE - DO NOT SHIP <<<")
            print("  Test completed without significance. Iterate or move on.")
    elif abs(relative_lift_pct) > 20 and days_elapsed < 3:
        print("  >>> CHECK FOR BUGS <<<")
        print("  Extreme early results are more likely instrumentation errors.")
        print("  Action: Audit logging, check randomization, verify metric definitions.")
    elif p_value < 0.05 and sample_fraction < 0.8:
        print("  >>> KEEP RUNNING <<<")
        print(f"  Only {sample_fraction:.0%} through planned duration.")
        print("  Risks of stopping now:")
        print("    - Novelty effect may be inflating results")
        print("    - Winner's curse: observed effect likely overstated")
        print(f"    - Peeking has inflated actual alpha above 0.05")
        print(f"    - Missing {'weekend' if days_elapsed < 7 else 'full weekly'} cycles")
    else:
        print("  >>> KEEP RUNNING <<<")
        print(f"  Test is {sample_fraction:.0%} complete. Let it run to full duration.")
    
    print(f"{'='*60}")

In [ ]:
# FamilyNest Scenarios

print("SCENARIO 1: PM's early peek (Day 5 of 21)")
print("─" * 60)
early_results_decision(
    p_value=0.03, 
    relative_lift_pct=8.0, 
    days_elapsed=5, 
    planned_days=21
)

print("\n\n")
print("SCENARIO 2: Guardrail alert -- host response rate dropping")
print("─" * 60)
early_results_decision(
    p_value=0.004, 
    relative_lift_pct=-15.0, 
    days_elapsed=7, 
    planned_days=21,
    is_guardrail_negative=True
)

print("\n\n")
print("SCENARIO 3: Test complete, barely significant")
print("─" * 60)
early_results_decision(
    p_value=0.048, 
    relative_lift_pct=3.2, 
    days_elapsed=21, 
    planned_days=21
)

print("\n\n")
print("SCENARIO 4: Suspicious extreme results on Day 1")
print("─" * 60)
early_results_decision(
    p_value=0.001, 
    relative_lift_pct=35.0, 
    days_elapsed=1, 
    planned_days=21
)

---
## Summary: Key Takeaways

### Rules of Thumb

1. **Never peek and decide** without pre-planned correction (alpha-spending or sequential testing)
2. **Novelty effects are real** -- always check if lift decays over time before trusting early results
3. **Winner's curse inflates effects** -- early significant results from underpowered tests overestimate the true effect
4. **p > 0.05 is not proof of no effect** -- it's absence of evidence, not evidence of absence
5. **Always report confidence intervals** -- they tell the story better than p-values alone
6. **Full weekly cycles matter** -- especially for FamilyNest where family booking behavior differs dramatically by day of week

### The FamilyNest Playbook

When pressured to ship early:
1. Acknowledge the exciting signal
2. Show the novelty effect analysis (lift over time)
3. Explain winner's curse in plain terms
4. Propose a concrete timeline ("2 more weeks and we'll know for sure")
5. Frame it as risk management: "Shipping early risks shipping something that actually hurts us"

In [ ]:
# Quick reference: all functions defined in this notebook

print("""
FUNCTIONS DEFINED IN THIS NOTEBOOK
===================================

1. analyze_novelty_effect(df, metric)
   - Hourly cohort analysis for novelty detection
   - Requires: 'hrs_to_cohort', 'variant', metric column

2. analyze_novelty_effect_days(df, metric)
   - Daily cohort analysis for longer experiments
   - Requires: 'days_in_experiment', 'variant', metric column
   - Includes automatic interpretation annotation

3. report_experiment_results(control_data, treatment_data, metric_name, mde_pct)
   - Full results summary with CI and interpretation
   - Use at end of experiment

4. early_results_decision(p_value, relative_lift_pct, days_elapsed, 
                          planned_days, is_guardrail_negative, sample_fraction)
   - Decision framework for mid-experiment situations
   - Use when stakeholders ask "can we ship?"

5. obrien_fleming_boundary(n_looks, alpha)
   - Calculate alpha-spending boundaries for planned interim analyses
   - Use when designing experiments with planned early looks
""")